# Support vector machines: the widest street
**Machine Learning · Week 4 (Kernels and support vector machines) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w4d2.html

A **support vector machine** draws the widest possible street between two classes.
It only needs *similarities* between points, so a **kernel** can replace the feature map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=60, centers=[(-1.5, -1), (1.5, 1)], cluster_std=0.8, random_state=2)
svm = SVC(kernel="linear", C=1.0).fit(X, y)
print("support vectors:", len(svm.support_vectors_), "of", len(X), "points")
print("street width   :", round(2 / np.linalg.norm(svm.coef_), 3))

In [ ]:
xx, yy = np.meshgrid(np.linspace(-4.5, 4.5, 300), np.linspace(-4, 4, 300))
f = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(layout="constrained")
ax.contourf(xx, yy, np.sign(f), levels=[-2, 0, 2], colors=["#ffe3d6", "#dfe6ff"])
ax.contour(xx, yy, f, levels=[-1, 0, 1], colors="#2c2842", linestyles=["--", "-", "--"], linewidths=[1.2, 2, 1.2])
ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "#ff8a50", "#5577f5"), s=30, zorder=3)
ax.scatter(*svm.support_vectors_.T, s=170, facecolors="none", edgecolors="#f0ac2e", linewidths=2, zorder=4, label="support vectors")
ax.legend(loc="upper left")
ax.set_title("The widest street, and the points that hold it")
plt.show()

Only the circled **support vectors** decide where the street goes. Move any other point a little and nothing changes.

## Soft margins: the knob C
Real data overlaps, so some points must sit inside the street (or on the wrong side). `C` sets the price
of each violation: small `C` = a wide, forgiving street; large `C` = "get every training point right".

In [ ]:
from sklearn.datasets import make_blobs as blobs

Xo, yo = blobs(n_samples=120, centers=[(-1, -0.6), (1, 0.6)], cluster_std=1.0, random_state=5)
for C in [0.01, 0.1, 1, 10, 100]:
    m = SVC(kernel="linear", C=C).fit(Xo, yo)
    print(f"C = {C:6g}: street width {2 / np.linalg.norm(m.coef_):.2f}, support vectors {len(m.support_):3d}, "
          f"training accuracy {m.score(Xo, yo):.3f}")

## Your turn: an SVM in PyTorch
A linear SVM is a linear model trained with the **hinge loss** plus an L2 penalty (week 2's ridge!):
$\frac1n\sum_i \max(0,\ 1 - y_i f(x_i)) + \lambda \lVert w\rVert^2$, with labels $y_i \in \{-1, +1\}$.
Vert^2$, with labels $y_i \in \{-1, +1\}$.
Points safely outside the street ($y f \ge 1$) cost nothing; that's why only support vectors matter.
Write the loss, train, and compare your direction with scikit-learn's.

In [ ]:
import torch

torch.manual_seed(0)
Xt = torch.tensor(Xo, dtype=torch.float32)
yt = torch.tensor(np.where(yo == 1, 1.0, -1.0), dtype=torch.float32)     # labels -1 / +1
model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
lam = 0.01
for step in range(2000):
    optimizer.zero_grad()
    f = model(Xt).squeeze(1)
    loss = f.sum() * 0           # TODO: the hinge loss plus lam * (model.weight ** 2).sum()
    loss.backward()
    optimizer.step()

w = model.weight.detach().numpy().ravel()
ref = SVC(kernel="linear", C=1 / (2 * lam * len(Xo))).fit(Xo, yo).coef_.ravel()
print("cosine between the two directions:", round(float(w @ ref / (np.linalg.norm(w) * np.linalg.norm(ref) + 1e-12)), 3))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
torch.manual_seed(0)
model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
for step in range(2000):
    optimizer.zero_grad()
    f = model(Xt).squeeze(1)
    loss = torch.clamp(1 - yt * f, min=0).mean() + lam * (model.weight ** 2).sum()
    loss.backward()
    optimizer.step()

w = model.weight.detach().numpy().ravel()
print("PyTorch direction:      ", np.round(w / np.linalg.norm(w), 3))
print("scikit-learn direction: ", np.round(ref / np.linalg.norm(ref), 3))
print("cosine between them:", round(float(w @ ref / (np.linalg.norm(w) * np.linalg.norm(ref))), 3), "(1 = same direction)")